# Feature Engineering

## Business Goal

Raw business data often lacks the features required for predictive machine learning.

Feature engineering transforms operational transaction data into informative variables that capture customer behavior, logistics efficiency, profitability, seasonality, and product performance.

The engineered features created in this notebook will be used for predictive modeling and factory optimization in later stages of the project.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
data_path = Path("../data/raw")

csv_file = list(data_path.glob("*.csv"))[0]

df = pd.read_csv(csv_file)

print("Dataset loaded successfully")
print(f"Dataset shape: {df.shape}")

Dataset loaded successfully
Dataset shape: (10194, 18)


## Date Feature Engineering

Date-based features help capture seasonality, quarterly business cycles, weekday purchasing behavior, and weekend demand patterns.

These features are useful for predictive modeling because raw date columns are not directly meaningful to most machine learning algorithms.

In [3]:
# Convert Order Date to datetime

df["Order Date"] = pd.to_datetime(
    df["Order Date"],
    errors="coerce"
)

# Create date-based features from Order Date

df["Order Year"] = df["Order Date"].dt.year
df["Order Month"] = df["Order Date"].dt.month
df["Order Month Name"] = df["Order Date"].dt.month_name()
df["Order Quarter"] = df["Order Date"].dt.quarter
df["Order Weekday"] = df["Order Date"].dt.day_name()

# Weekend flag: Saturday = 5, Sunday = 6

df["Is Weekend"] = (
    df["Order Date"].dt.dayofweek >= 5
).astype(int)

df[[
    "Order Date",
    "Order Year",
    "Order Month",
    "Order Month Name",
    "Order Quarter",
    "Order Weekday",
    "Is Weekend"
]].head()

,Order Date,Order Year,Order Month,Order Month Name,Order Quarter,Order Weekday,Is Weekend
0,2024-03-01,2024.0,3.0,March,1.0,Friday,0
1,2024-04-01,2024.0,4.0,April,2.0,Monday,0
2,2024-04-01,2024.0,4.0,April,2.0,Monday,0
3,2024-04-01,2024.0,4.0,April,2.0,Monday,0
4,2024-05-01,2024.0,5.0,May,2.0,Wednesday,0


### Observation

Six date-based features were created from the Order Date column.

These include year, month, month name, quarter, weekday, and weekend indicator.

### Business Interpretation

Temporal features allow machine learning models to capture seasonal purchasing behavior and business cycles.

These features can help improve demand forecasting, production planning, and inventory optimization.

## Delivery Feature Engineering

### Data Quality Note

The Ship Date values in the dataset produce unrealistic delivery durations when compared with Order Date.

For example, some records show shipment dates that are hundreds of days after the order date, which does not represent normal business operations for candy distribution.

To avoid introducing misleading features into the machine learning pipeline, delivery-based features such as delivery days, delivery speed, and late delivery flag are intentionally excluded from this project.

## Financial Feature Engineering

Financial features help measure profitability, cost efficiency, and order value.

These engineered variables are useful for predicting business performance and supporting factory allocation decisions.

In [4]:
# Profit margin percentage

df["Profit Margin (%)"] = (
    df["Gross Profit"] / df["Sales"]
) * 100

# Profit generated per unit sold

df["Profit Per Unit"] = (
    df["Gross Profit"] / df["Units"]
)

# Cost as a percentage of sales

df["Cost Percentage (%)"] = (
    df["Cost"] / df["Sales"]
) * 100

# High value order flag based on top 25% sales value

high_value_threshold = df["Sales"].quantile(0.75)

df["High Value Order"] = (
    df["Sales"] >= high_value_threshold
).astype(int)

# Sales category

df["Sales Category"] = pd.qcut(
    df["Sales"],
    q=3,
    labels=["Low", "Medium", "High"]
)

# Profit category

df["Profit Category"] = pd.qcut(
    df["Gross Profit"],
    q=3,
    labels=["Low", "Medium", "High"]
)

df[[
    "Sales",
    "Gross Profit",
    "Cost",
    "Units",
    "Profit Margin (%)",
    "Profit Per Unit",
    "Cost Percentage (%)",
    "High Value Order",
    "Sales Category",
    "Profit Category"
]].head()

,Sales,Gross Profit,Cost,Units,Profit Margin (%),Profit Per Unit,Cost Percentage (%),High Value Order,Sales Category,Profit Category
0,6.50,4.22,2.28,2,64.923077,2.11,35.076923,0,Low,Low
1,7.50,4.90,2.60,2,65.333333,2.45,34.666667,0,Low,Low
2,10.47,7.47,3.00,3,71.346705,2.49,28.653295,0,Medium,Medium
3,10.80,7.50,3.30,3,69.444444,2.50,30.555556,0,Medium,Medium
4,11.25,7.35,3.90,3,65.333333,2.45,34.666667,0,Medium,Medium


### Observation

Six financial features were created to capture profitability, cost efficiency, and order value.

These include profit margin, profit per unit, cost percentage, high-value order flag, sales category, and profit category.

### Business Interpretation

Financial features help the model understand not only how much revenue an order generates, but also how efficiently it contributes to profit.

These variables are important for optimization because factory reassignment decisions should improve logistics performance without harming profitability.

## Save Engineered Dataset

The dataset is saved after feature engineering so it can be used directly during predictive modeling.

Saving an intermediate processed dataset improves reproducibility and separates data preparation from machine learning workflows.

In [5]:
from pathlib import Path

# Create processed data folder

processed_path = Path("../data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

# Save dataset

output_file = processed_path / "factory_sales_feature_engineered.csv"

df.to_csv(output_file, index=False)

print("Feature engineered dataset saved successfully.")
print(f"Location: {output_file}")
print(f"Shape: {df.shape}")

Feature engineered dataset saved successfully.
Location: ..\data\processed\factory_sales_feature_engineered.csv
Shape: (10194, 30)


# Conclusion

Feature engineering transformed the cleaned transactional dataset into a richer analytical dataset suitable for machine learning.

The notebook introduced temporal and financial features that capture purchasing behavior, profitability, and business performance.

Delivery-related features were intentionally excluded because the dataset contained inconsistent shipment timestamps that produced unrealistic delivery durations. Excluding unreliable variables helps improve model quality and demonstrates sound data science practice.

The engineered dataset will be used in the next stage for predictive modeling and factory optimization.